In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset

# -----------------------
# setup
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)

# -----------------------
# data: sklearn digits
# -----------------------
digits = load_digits()
x = digits.data.astype("float32") / 16.0
y = digits.target

x_tensor = torch.tensor(x)
dataset = TensorDataset(x_tensor)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

# -----------------------
# Autoencoder
# -----------------------
latent_dim = 2

class AutoEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        x_rec = self.decode(z)
        return x_rec, z

ae = AutoEncoder(latent_dim=latent_dim).to(device)
opt_ae = torch.optim.Adam(ae.parameters(), lr=1e-3)

# -----------------------
# train Autoencoder
# -----------------------
for epoch in range(100):
    total = 0.0

    for (xb,) in loader:
        xb = xb.to(device)

        x_rec, z = ae(xb)
        loss = ((x_rec - xb) ** 2).mean()

        opt_ae.zero_grad()
        loss.backward()
        opt_ae.step()

        total += loss.item()

    if epoch % 20 == 0:
        print("AE epoch", epoch, "loss", total / len(loader))

# -----------------------
# latent dataset
# -----------------------
ae.eval()

with torch.no_grad():
    z_data = ae.encode(x_tensor.to(device)).cpu()

z_mean = z_data.mean(dim=0, keepdim=True)
z_std = z_data.std(dim=0, keepdim=True)

z_data_norm = (z_data - z_mean) / z_std

latent_dataset = TensorDataset(z_data_norm)
latent_loader = DataLoader(latent_dataset, batch_size=256, shuffle=True)

# -----------------------
# Diffusion schedule
# -----------------------
T = 100

betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# -----------------------
# Diffusion model in latent space
# eps_theta(z_t, t)
# -----------------------
class LatentDiffusionModel(nn.Module):
    def __init__(self, latent_dim=2, hidden=128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, latent_dim),
        )

    def forward(self, z_t, t):
        # t: [batch], integer timestep
        t = t.float().view(-1, 1) / T
        return self.net(torch.cat([z_t, t], dim=1))

diffusion = LatentDiffusionModel(latent_dim=latent_dim).to(device)
opt_diff = torch.optim.Adam(diffusion.parameters(), lr=1e-3)

# -----------------------
# forward diffusion q(z_t | z_0)
# -----------------------
def q_sample(z0, t, noise):
    alpha_bar_t = alpha_bars[t].view(-1, 1)

    z_t = (
        torch.sqrt(alpha_bar_t) * z0
        + torch.sqrt(1.0 - alpha_bar_t) * noise
    )

    return z_t

# -----------------------
# train latent diffusion
# -----------------------
for epoch in range(500):
    total = 0.0

    for (z0,) in latent_loader:
        z0 = z0.to(device)
        batch_size = z0.shape[0]

        t = torch.randint(0, T, (batch_size,), device=device)
        noise = torch.randn_like(z0)

        z_t = q_sample(z0, t, noise)

        noise_pred = diffusion(z_t, t)
        loss = ((noise_pred - noise) ** 2).mean()

        opt_diff.zero_grad()
        loss.backward()
        opt_diff.step()

        total += loss.item()

    if epoch % 50 == 0:
        print("Diffusion epoch", epoch, "loss", total / len(latent_loader))

# -----------------------
# reverse diffusion sampling with path recording
# -----------------------
@torch.no_grad()
def sample_latent_paths_diffusion(model, n_samples=5000):
    model.eval()

    z = torch.randn(n_samples, latent_dim, device=device)

    # paths[:, 0] = pure noise at t=T
    paths = torch.zeros(n_samples, T + 1, latent_dim, device=device)
    paths[:, 0] = z

    step = 1

    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)

        beta_t = betas[t]
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]

        eps_pred = model(z, t_batch)

        mean = (
            1.0 / torch.sqrt(alpha_t)
            * (
                z
                - beta_t / torch.sqrt(1.0 - alpha_bar_t) * eps_pred
            )
        )

        if t > 0:
            noise = torch.randn_like(z)
            z = mean + torch.sqrt(beta_t) * noise
        else:
            z = mean

        paths[:, step] = z
        step += 1

    return paths.cpu()

paths = sample_latent_paths_diffusion(diffusion, n_samples=10000)

# -----------------------
# generate digit images
# -----------------------
@torch.no_grad()
def generate_images(diffusion, ae, n_samples=64):
    diffusion.eval()
    ae.eval()

    z = torch.randn(n_samples, latent_dim, device=device)

    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)

        beta_t = betas[t]
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]

        eps_pred = diffusion(z, t_batch)

        mean = (
            1.0 / torch.sqrt(alpha_t)
            * (
                z
                - beta_t / torch.sqrt(1.0 - alpha_bar_t) * eps_pred
            )
        )

        if t > 0:
            noise = torch.randn_like(z)
            z = mean + torch.sqrt(beta_t) * noise
        else:
            z = mean

    # unnormalize latent
    z = z.cpu() * z_std + z_mean
    z = z.to(device)

    x_gen = ae.decode(z)
    return x_gen.cpu().numpy()

imgs = generate_images(diffusion, ae, n_samples=64)

fig, axes = plt.subplots(8, 8, figsize=(6, 6))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(imgs[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")

plt.suptitle("Generated digits by Autoencoder Latent Diffusion Model")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------
# Path Density: z1 direction only
# -----------------------
z_index = 0
n_steps = paths.shape[1] - 1


In [ ]:
# -----------------------
# 2D latent density 
# -----------------------

# t = 0, 0.1, ..., 1.0 に対応する index
show_indices = [int(n_steps * k / 10) for k in range(11)]
show_times = [k / 10 for k in range(11)]

z_min = -4
z_max = 4

fig, axes = plt.subplots(
    3, 4,
    figsize=(12, 9),
    sharex=True,
    sharey=True
)

axes = axes.ravel()

# velocity field grid
grid_size = 15
z1_grid = np.linspace(z_min, z_max, grid_size)
z2_grid = np.linspace(z_min, z_max, grid_size)
Z1, Z2 = np.meshgrid(z1_grid, z2_grid)

z_grid_tensor = torch.tensor(
    np.stack([Z1.reshape(-1), Z2.reshape(-1)], axis=1),
    dtype=torch.float32,
    device=device
)

for ax, idx, tval in zip(axes, show_indices, show_times):
    z = paths[:, idx, :].numpy()

    ax.hist2d(
        z[:, 0],
        z[:, 1],
        bins=80,
        range=[[z_min, z_max], [z_min, z_max]],
        cmap="viridis"
    )

    t_tensor = torch.full(
        (z_grid_tensor.shape[0], 1),
        tval,
        dtype=torch.float32,
        device=device
    )

    ax.set_title(f"t = {tval:.1f}")
    ax.set_xlim(z_min, z_max)
    ax.set_ylim(z_min, z_max)
    ax.set_aspect("equal")
    ax.set_xlabel("z1")
    ax.set_ylabel("z2")

for j in range(len(show_indices), len(axes)):
    axes[j].axis("off")

plt.suptitle("2D Latent Path Density")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------
# Decode 10x10 grid on t=1 latent space
# -----------------------

@torch.no_grad()
def plot_decoder_grid(ae, z_mean, z_std, z_min=-4, z_max=4, grid_size=10):
    ae.eval()

    z1_values = np.linspace(z_min, z_max, grid_size)
    z2_values = np.linspace(z_min, z_max, grid_size)

    fig, axes = plt.subplots(
        grid_size,
        grid_size,
        figsize=(10, 10),
        sharex=True,
        sharey=True
    )

    for i, z2 in enumerate(z2_values[::-1]):
        for j, z1 in enumerate(z1_values):
            # normalized latent coordinate
            z_norm = torch.tensor(
                [[z1, z2]],
                dtype=torch.float32,
                device=device
            )

            # unnormalize latent because decoder was trained on original AE latent
            z = z_norm.cpu() * z_std + z_mean
            z = z.to(device)

            x_dec = ae.decode(z)
            img = x_dec.cpu().numpy().reshape(8, 8)

            ax = axes[i, j]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"{z1:.1f},{z2:.1f}", fontsize=7)
            ax.axis("off")

    plt.suptitle("Decoder output on 10x10 latent grid at t=1")
    plt.tight_layout()
    plt.show()


plot_decoder_grid(
    ae=ae,
    z_mean=z_mean,
    z_std=z_std,
    z_min=-2,
    z_max=2,
    grid_size=10
)